# 🛡️ GUARANTEED Historical Data Export — One District at a Time

## Why This Notebook Exists

Previous approaches failed because:
- **getInfo() / computePixels()** — 10 MB request-payload limit when sending 64 complex district polygons
- **Export.table.toDrive() with ALL districts** — ALSO hit 10 MB limit (64 polygon geometries in one request)

## The Bulletproof Solution

**Export ONE district at a time.** Each export task sends exactly **one** polygon to GEE.

| Metric | Value |
|--------|-------|
| Districts | 64 |
| Export tasks per dataset | 64 (one per district) |
| Payload per task | ~5–50 KB (TINY) |
| Can hit 10 MB limit? | **IMPOSSIBLE** |

## What Data Do We Need?

| Dataset | Source | Variables | Used For |
|---------|--------|-----------|----------|
| **CHIRPS** | `UCSB-CHG/CHIRPS/DAILY` | `precipitation` (mm) | SPI computation |
| **ERA5-Land** | `ECMWF/ERA5_LAND/DAILY_AGGR` | `temperature_2m` (K→°C), `total_evaporation_sum` (m→mm) | SPEI computation |

These are the **ONLY two raw historical sources** needed:
- **CHIRPS → SPI** (Standardized Precipitation Index)
- **ERA5-Land + CHIRPS → SPEI** (Standardized Precipitation-Evapotranspiration Index)

### Two Modes

| Mode | Years | Approx. Time | Use Case |
|------|-------|-------------|----------|
| `recent` | 2020–2023 | ~30 min per dataset | Quick test / demo |
| `full` | 1981–2023 | ~2–4 hours per dataset | Full climatology baseline |

### Workflow

1. **Cells 1–3** → Setup, configure, load districts
2. **Cells 4–4b** → Submit & monitor CHIRPS exports (64 tasks)
3. **Cells 5–5b** → Submit & monitor ERA5-Land exports (64 tasks)
4. **Wait** → Monitor at https://code.earthengine.google.com/tasks
5. **Download** → Get CSVs from Google Drive
6. **Cells 6–8** → Combine CHIRPS & compute climatology
7. **Cells 9–10** → Combine ERA5 & compute climatology
8. **Cell 11** → Final summary & verification

---

## Cell 1 — Install Dependencies & Authenticate GEE

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 1: Install dependencies & authenticate GEE
# ══════════════════════════════════════════════════════════════════════

!pip install -q earthengine-api geopandas shapely pandas

import ee

# ── Authenticate (only needed once per Colab session) ──
try:
    ee.Initialize()
    print('✅ GEE already authenticated')
except Exception:
    ee.Authenticate()
    ee.Initialize()
    print('✅ GEE authenticated successfully')

# Verify connection
test = ee.Image('USGS/SRTMGL1_003').getInfo()
print(f'✅ GEE connection verified (SRTM image type: {test["type"]})')

## Cell 2 — Choose Mode & Configure

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 2: Choose your mode
# ══════════════════════════════════════════════════════════════════════

# ┌──────────────────────────────────────────────────────────────┐
# │  CHANGE THIS to 'full' for complete 1981-2023 climatology   │
# │  CHANGE THIS to 'recent' for quick 2020-2023 test           │
# └──────────────────────────────────────────────────────────────┘
YEARS_MODE = 'recent'  # Options: 'full' or 'recent'

# Google Drive folder name where exports will land
DRIVE_FOLDER = 'bangladesh_drought_historical'

# ── Derive year range ─────────────────────────────────────────
if YEARS_MODE == 'full':
    START_YEAR = 1981
    END_YEAR = 2023
elif YEARS_MODE == 'recent':
    START_YEAR = 2020
    END_YEAR = 2023
else:
    raise ValueError(f"Unknown YEARS_MODE: {YEARS_MODE}. Use 'full' or 'recent'.")

print(f'📅 Mode:       {YEARS_MODE}')
print(f'📅 Year range: {START_YEAR}–{END_YEAR} ({END_YEAR - START_YEAR + 1} years)')
print(f'📁 Drive folder: {DRIVE_FOLDER}')
print()
print('⚠️  Each district gets its OWN export task (64 total).')
print('    Each task payload is tiny — CANNOT hit the 10 MB limit.')

## Cell 3 — Load District Boundaries

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 3: Load district boundaries
# ══════════════════════════════════════════════════════════════════════

import sys, os
import pandas as pd
import geopandas as gpd
from shapely import wkt
from pathlib import Path

# ── Try loading from production_pipeline ──
districts_gdf = None

# Method 1: From production_pipeline bundled data
try:
    # If running from the repo root
    sys.path.insert(0, str(Path.cwd()))
    from production_pipeline.extractors.static_extractor import load_hdx_boundaries
    districts_gdf = load_hdx_boundaries()
    # Ensure canonical columns exist
    if 'district_id_canonical' not in districts_gdf.columns:
        districts_gdf['district_id_canonical'] = districts_gdf['district_id']
    if 'district_name_canonical' not in districts_gdf.columns:
        districts_gdf['district_name_canonical'] = districts_gdf['district_name']
    print(f'✅ Loaded {len(districts_gdf)} districts from production_pipeline')
except Exception as e:
    print(f'⚠️  Could not load from production_pipeline: {e}')

# Method 2: Direct CSV load (if production_pipeline import fails)
if districts_gdf is None:
    # Try common paths
    candidate_paths = [
        Path('production_pipeline/data/static/hdx_boundaries.csv'),
        Path('/home/ubuntu/production_pipeline/data/static/hdx_boundaries.csv'),
        Path('/content/production_pipeline/data/static/hdx_boundaries.csv'),
        Path('/content/hdx_boundaries.csv'),
    ]
    for p in candidate_paths:
        if p.exists():
            df = pd.read_csv(p, dtype={'district_id': str})
            if 'geometry_wkt' in df.columns:
                df['geometry'] = df['geometry_wkt'].apply(
                    lambda g: wkt.loads(g) if pd.notna(g) and g else None
                )
                districts_gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')
            else:
                districts_gdf = df
            # Ensure canonical columns
            districts_gdf['district_id_canonical'] = districts_gdf['district_id']
            districts_gdf['district_name_canonical'] = districts_gdf['district_name']
            print(f'✅ Loaded {len(districts_gdf)} districts from {p}')
            break

if districts_gdf is None:
    raise FileNotFoundError(
        '❌ Could not find hdx_boundaries.csv!\n'
        'Please upload it to /content/ or ensure production_pipeline/data/static/ exists.\n'
        'You can get it from the production_pipeline repo.'
    )

# ── Verify ──
assert len(districts_gdf) == 64, f'Expected 64 districts, got {len(districts_gdf)}'
assert 'geometry' in districts_gdf.columns or 'geometry_wkt' in districts_gdf.columns

print(f'\n📊 District summary:')
print(f'   Total: {len(districts_gdf)} districts')
print(f'   Columns: {list(districts_gdf.columns)}')
print(f'   Sample IDs: {districts_gdf["district_id_canonical"].head(5).tolist()}')
print(f'   Sample names: {districts_gdf["district_name_canonical"].head(5).tolist()}')

## Cell 4 — Export CHIRPS: One District at a Time (GUARANTEED)

This cell submits **64 independent export tasks** to GEE. Each task:
- Sends **ONE** polygon (one district)
- Covers all selected years
- Exports to Google Drive as CSV
- Payload is ~5–50 KB — **physically impossible** to hit 10 MB

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 4: Submit 64 export tasks — ONE district per task
# ══════════════════════════════════════════════════════════════════════

import ee
import time
from shapely import wkt as shapely_wkt
from shapely.geometry import MultiPolygon, Polygon


def _shapely_to_ee_geometry(geom_shapely):
    """Convert a Shapely geometry to an ee.Geometry.
    
    Handles both Polygon and MultiPolygon.
    For MultiPolygon, uses the largest polygon by area.
    """
    if isinstance(geom_shapely, MultiPolygon):
        # Use the largest polygon to keep payload small
        geom_shapely = max(geom_shapely.geoms, key=lambda g: g.area)
    
    # Simplify geometry to reduce coordinate count (tolerance in degrees ~100m)
    geom_simplified = geom_shapely.simplify(tolerance=0.005, preserve_topology=True)
    
    coords = list(geom_simplified.exterior.coords)
    return ee.Geometry.Polygon(coords)


def export_chirps_one_district(district_row, start_year, end_year, drive_folder):
    """Export CHIRPS daily rainfall for ONE district across all specified years.
    
    Returns the export task object.
    
    Guaranteed to work because:
    - Only 1 polygon in the request (~5-50 KB payload)
    - Cannot hit 10 MB limit
    """
    district_id = district_row['district_id_canonical']
    district_name = district_row['district_name_canonical']
    
    # ── Parse geometry ──
    if hasattr(district_row, 'geometry') and district_row['geometry'] is not None:
        geom_shapely = district_row['geometry']
        if isinstance(geom_shapely, str):
            geom_shapely = shapely_wkt.loads(geom_shapely)
    elif 'geometry_wkt' in district_row.index and pd.notna(district_row['geometry_wkt']):
        geom_shapely = shapely_wkt.loads(district_row['geometry_wkt'])
    else:
        raise ValueError(f'No geometry found for district {district_id}')
    
    # ── Convert to EE geometry ──
    ee_geom = _shapely_to_ee_geometry(geom_shapely)
    
    # ── CHIRPS collection for the entire date range ──
    date_start = f'{start_year}-01-01'
    date_end = f'{end_year + 1}-01-01'  # exclusive end
    
    chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
              .filterDate(date_start, date_end)
              .filterBounds(ee_geom))
    
    # ── Reduce each daily image to district mean ──
    def reduce_image(img):
        date_str = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
        stats = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=ee_geom,
            scale=5566,        # CHIRPS native resolution
            maxPixels=1e9,
            bestEffort=True
        )
        return ee.Feature(None, {
            'district_id': district_id,
            'district_name': district_name,
            'date': date_str,
            'rainfall_mm': stats.get('precipitation')
        })
    
    results = chirps.map(reduce_image)
    
    # ── Export to Drive ──
    safe_name = district_id.replace(' ', '_').replace('-', '_')
    task_desc = f'chirps_{safe_name}_{start_year}_{end_year}'
    
    task = ee.batch.Export.table.toDrive(
        collection=results,
        description=task_desc,
        folder=drive_folder,
        fileNamePrefix=f'chirps_{safe_name}',
        fileFormat='CSV',
        selectors=['district_id', 'district_name', 'date', 'rainfall_mm']
    )
    task.start()
    return task, task_desc


# ══════════════════════════════════════════════════════════════════════
# Submit all 64 exports
# ══════════════════════════════════════════════════════════════════════

print('=' * 80)
print(f'🚀 SUBMITTING CHIRPS EXPORTS — ONE DISTRICT AT A TIME')
print(f'📅 Years: {START_YEAR}–{END_YEAR} ({END_YEAR - START_YEAR + 1} years)')
print(f'📁 Google Drive folder: {DRIVE_FOLDER}')
print(f'📊 Districts: {len(districts_gdf)}')
print(f'📋 Total tasks: {len(districts_gdf)} (one per district)')
print('=' * 80)
print()

tasks = []
failed = []

for idx, row in districts_gdf.iterrows():
    did = row['district_id_canonical']
    dname = row['district_name_canonical']
    try:
        task, task_desc = export_chirps_one_district(row, START_YEAR, END_YEAR, DRIVE_FOLDER)
        tasks.append({'district_id': did, 'district_name': dname, 'task_desc': task_desc, 'task': task})
        print(f'  ✅ [{len(tasks):02d}/64] {dname} ({did})')
        
        # Small delay to avoid rate limiting
        if len(tasks) % 10 == 0:
            time.sleep(2)
            
    except Exception as e:
        failed.append({'district_id': did, 'district_name': dname, 'error': str(e)})
        print(f'  ❌ [{len(tasks) + len(failed):02d}/64] {dname} ({did}): {e}')

# ── Summary ──
print()
print('=' * 80)
print(f'📋 EXPORT SUMMARY')
print(f'   ✅ Submitted: {len(tasks)} tasks')
print(f'   ❌ Failed:    {len(failed)} tasks')
print('=' * 80)

if failed:
    print('\n⚠️  Failed districts:')
    for f in failed:
        print(f'   - {f["district_name"]} ({f["district_id"]}): {f["error"]}')

print(f'\n📌 NEXT STEPS:')
print(f'   1. Monitor tasks: https://code.earthengine.google.com/tasks')
print(f'   2. Wait for ALL {len(tasks)} tasks to complete (green checkmarks)')
print(f'   3. Go to Google Drive → {DRIVE_FOLDER}/')
print(f'   4. Download all {len(tasks)} CSV files')
print(f'   5. Upload them to /content/chirps_by_district/ (or wherever convenient)')
print(f'   6. Run the NEXT cells to combine and compute climatology')

## Cell 4b — (Optional) Monitor Export Progress

Run this cell periodically to check how many tasks are done.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 4b: Monitor task progress (run periodically)
# ══════════════════════════════════════════════════════════════════════

from collections import Counter

statuses = []
for t in tasks:
    status = t['task'].status()
    state = status.get('state', 'UNKNOWN')
    statuses.append(state)

counts = Counter(statuses)
total = len(tasks)
completed = counts.get('COMPLETED', 0)
running = counts.get('RUNNING', 0)
ready = counts.get('READY', 0)
failed_count = counts.get('FAILED', 0)

pct = (completed / total * 100) if total > 0 else 0
bar = '█' * int(pct // 2) + '░' * (50 - int(pct // 2))

print(f'Progress: [{bar}] {pct:.0f}%')
print(f'  ✅ COMPLETED: {completed}/{total}')
print(f'  🔄 RUNNING:   {running}')
print(f'  ⏳ READY:     {ready}')
print(f'  ❌ FAILED:    {failed_count}')

if failed_count > 0:
    print('\n⚠️  Failed tasks:')
    for t in tasks:
        s = t['task'].status()
        if s.get('state') == 'FAILED':
            print(f'   - {t["district_name"]}: {s.get("error_message", "unknown error")}')

if completed == total:
    print('\n🎉 ALL DONE! Proceed to download from Google Drive and run the next cells.')

---

## Cell 5 — Export ERA5-Land: One District at a Time (GUARANTEED)

Same bulletproof approach as CHIRPS. Each task sends **one** polygon.

| Variable | ERA5 Band | Unit Conversion | Used For |
|----------|-----------|-----------------|----------|
| Temperature | `temperature_2m` | Kelvin → Celsius | SPEI (PET estimation) |
| Evapotranspiration | `total_evaporation_sum` | meters → mm | SPEI (water balance) |

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 5: Submit 64 ERA5-Land export tasks — ONE district per task
# ══════════════════════════════════════════════════════════════════════

import ee
import time
from shapely import wkt as shapely_wkt
from shapely.geometry import MultiPolygon, Polygon


def export_era5_one_district(district_row, start_year, end_year, drive_folder):
    """Export ERA5-Land for ONE district, all years.
    
    Variables: temperature_2m, total_evaporation_sum (for SPEI)
    Guaranteed to work — tiny payload (just 1 district!).
    
    Returns
    -------
    tuple(task, task_desc)
    """
    district_id = district_row['district_id_canonical']
    district_name = district_row['district_name_canonical']
    
    # ── Parse geometry ──
    if hasattr(district_row, 'geometry') and district_row['geometry'] is not None:
        geom_shapely = district_row['geometry']
        if isinstance(geom_shapely, str):
            geom_shapely = shapely_wkt.loads(geom_shapely)
    elif 'geometry_wkt' in district_row.index and pd.notna(district_row['geometry_wkt']):
        geom_shapely = shapely_wkt.loads(district_row['geometry_wkt'])
    else:
        raise ValueError(f'No geometry found for district {district_id}')
    
    # ── Convert to EE geometry (reuse helper from CHIRPS cell) ──
    ee_geom = _shapely_to_ee_geometry(geom_shapely)
    
    # ── ERA5-Land collection ──
    date_start = f'{start_year}-01-01'
    date_end = f'{end_year + 1}-01-01'  # exclusive end
    
    era5 = (ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
            .filterDate(date_start, date_end)
            .filterBounds(ee_geom)
            .select(['temperature_2m', 'total_evaporation_sum']))
    
    # ── Reduce each daily image to district mean ──
    def reduce_image(img):
        date_str = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
        
        stats = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=ee_geom,
            scale=11132,  # ERA5-Land native resolution (~11 km)
            maxPixels=1e9,
            bestEffort=True
        )
        
        # Convert units server-side
        temp_k = ee.Number(stats.get('temperature_2m'))
        temp_c = temp_k.subtract(273.15)  # Kelvin → Celsius
        
        evap_m = ee.Number(stats.get('total_evaporation_sum'))
        evap_mm = evap_m.multiply(1000)  # meters → millimeters
        
        return ee.Feature(None, {
            'district_id': district_id,
            'district_name': district_name,
            'date': date_str,
            'temperature_c': temp_c,
            'evaporation_mm': evap_mm
        })
    
    results = era5.map(reduce_image)
    
    # ── Export to Drive ──
    safe_name = district_id.replace(' ', '_').replace('-', '_')
    task_desc = f'era5_{safe_name}_{start_year}_{end_year}'
    
    task = ee.batch.Export.table.toDrive(
        collection=results,
        description=task_desc,
        folder=drive_folder,
        fileNamePrefix=f'era5_{safe_name}',
        fileFormat='CSV',
        selectors=['district_id', 'district_name', 'date', 'temperature_c', 'evaporation_mm']
    )
    task.start()
    return task, task_desc


# ══════════════════════════════════════════════════════════════════════
# Submit all 64 ERA5 exports
# ══════════════════════════════════════════════════════════════════════

print('=' * 80)
print(f'🚀 SUBMITTING ERA5-LAND EXPORTS — ONE DISTRICT AT A TIME')
print(f'📅 Years: {START_YEAR}–{END_YEAR} ({END_YEAR - START_YEAR + 1} years)')
print(f'📁 Google Drive folder: {DRIVE_FOLDER}')
print(f'📊 Districts: {len(districts_gdf)}')
print(f'📋 Total tasks: {len(districts_gdf)} (one per district)')
print(f'📍 Variables: temperature_2m (K→°C), total_evaporation_sum (m→mm)')
print('=' * 80)
print()

era5_tasks = []
era5_failed = []

for idx, row in districts_gdf.iterrows():
    did = row['district_id_canonical']
    dname = row['district_name_canonical']
    try:
        task, task_desc = export_era5_one_district(row, START_YEAR, END_YEAR, DRIVE_FOLDER)
        era5_tasks.append({'district_id': did, 'district_name': dname, 'task_desc': task_desc, 'task': task})
        print(f'  ✅ [{len(era5_tasks):02d}/64] {dname} ({did})')
        
        # Small delay to avoid rate limiting
        if len(era5_tasks) % 10 == 0:
            time.sleep(2)
            
    except Exception as e:
        era5_failed.append({'district_id': did, 'district_name': dname, 'error': str(e)})
        print(f'  ❌ [{len(era5_tasks) + len(era5_failed):02d}/64] {dname} ({did}): {e}')

# ── Summary ──
print()
print('=' * 80)
print(f'📋 ERA5-LAND EXPORT SUMMARY')
print(f'   ✅ Submitted: {len(era5_tasks)} tasks')
print(f'   ❌ Failed:    {len(era5_failed)} tasks')
print('=' * 80)

if era5_failed:
    print('\n⚠️  Failed districts:')
    for f in era5_failed:
        print(f'   - {f["district_name"]} ({f["district_id"]}): {f["error"]}')

print(f'\n📌 NEXT STEPS:')
print(f'   1. Monitor tasks: https://code.earthengine.google.com/tasks')
print(f'   2. Wait for ALL {len(era5_tasks)} tasks to complete (green checkmarks)')
print(f'   3. Files will appear in Google Drive → {DRIVE_FOLDER}/')
print(f'   4. Files start with "era5_" (vs CHIRPS files that start with "chirps_")')

## Cell 5b — (Optional) Monitor ERA5 Export Progress

Run this cell periodically to check how many ERA5 tasks are done.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 5b: Monitor ERA5 task progress (run periodically)
# ══════════════════════════════════════════════════════════════════════

from collections import Counter

era5_statuses = []
for t in era5_tasks:
    status = t['task'].status()
    state = status.get('state', 'UNKNOWN')
    era5_statuses.append(state)

era5_counts = Counter(era5_statuses)
era5_total = len(era5_tasks)
era5_completed = era5_counts.get('COMPLETED', 0)
era5_running = era5_counts.get('RUNNING', 0)
era5_ready = era5_counts.get('READY', 0)
era5_failed_count = era5_counts.get('FAILED', 0)

pct = (era5_completed / era5_total * 100) if era5_total > 0 else 0
bar = '█' * int(pct // 2) + '░' * (50 - int(pct // 2))

print(f'ERA5-Land Progress: [{bar}] {pct:.0f}%')
print(f'  ✅ COMPLETED: {era5_completed}/{era5_total}')
print(f'  🔄 RUNNING:   {era5_running}')
print(f'  ⏳ READY:     {era5_ready}')
print(f'  ❌ FAILED:    {era5_failed_count}')

if era5_failed_count > 0:
    print('\n⚠️  Failed tasks:')
    for t in era5_tasks:
        s = t['task'].status()
        if s.get('state') == 'FAILED':
            print(f'   - {t["district_name"]}: {s.get("error_message", "unknown error")}')

if era5_completed == era5_total:
    print('\n🎉 ALL ERA5 TASKS DONE! Proceed to download from Google Drive.')

---

## ⬇️ PAUSE HERE — Wait for ALL Exports to Complete (CHIRPS + ERA5)

1. Go to https://code.earthengine.google.com/tasks
2. Wait until all **128 tasks** show ✅ COMPLETED (64 CHIRPS + 64 ERA5)
3. Go to Google Drive → `bangladesh_drought_historical/`
4. Download all CSV files:
   - `chirps_*.csv` → 64 files for CHIRPS rainfall
   - `era5_*.csv` → 64 files for ERA5 temperature & evaporation
5. Upload CHIRPS CSVs to `/content/chirps_by_district/`
6. Upload ERA5 CSVs to `/content/era5_by_district/`
7. Then run the cells below to combine and compute climatology

---

## Cell 6 — Combine CHIRPS District CSVs

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 6: Combine downloaded district CSVs into one dataset
# ══════════════════════════════════════════════════════════════════════

import pandas as pd
import glob
from pathlib import Path

# ┌─────────────────────────────────────────────────────────────────┐
# │  Option A: Read directly from Google Drive (if mounted)        │
# │  Option B: Read from uploaded files in /content/               │
# │  CHANGE THE PATH BELOW to match where your files are!          │
# └─────────────────────────────────────────────────────────────────┘

# Try multiple common locations
search_dirs = [
    '/content/drive/MyDrive/bangladesh_drought_historical',  # Google Drive mounted
    '/content/chirps_by_district',                            # Manual upload
    '/content/bangladesh_drought_historical',                 # Direct folder
    './chirps_by_district',                                   # Local
]

csv_files = []
source_dir = None
for d in search_dirs:
    found = sorted(glob.glob(f'{d}/chirps_*.csv'))
    if found:
        csv_files = found
        source_dir = d
        break

if not csv_files:
    # Try mounting Google Drive
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        csv_files = sorted(glob.glob(f'/content/drive/MyDrive/{DRIVE_FOLDER}/chirps_*.csv'))
        source_dir = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
    except Exception:
        pass

if not csv_files:
    raise FileNotFoundError(
        '❌ No CHIRPS CSV files found!\n'
        'Options:\n'
        '  1. Mount Google Drive: from google.colab import drive; drive.mount("/content/drive")\n'
        f'  2. Upload CSVs to /content/chirps_by_district/\n'
        f'  3. Check that exports completed at https://code.earthengine.google.com/tasks'
    )

print(f'📂 Source: {source_dir}')
print(f'📄 Found {len(csv_files)} CSV files')
print()

# ── Load and combine ──
all_data = []
errors = []

for f in csv_files:
    try:
        df = pd.read_csv(f)
        if len(df) == 0:
            print(f'  ⚠️  Empty: {Path(f).name}')
            continue
        all_data.append(df)
        district_id = df['district_id'].iloc[0] if 'district_id' in df.columns else 'unknown'
        print(f'  ✅ {Path(f).name}: {len(df):>8,} rows  ({district_id})')
    except Exception as e:
        errors.append(f)
        print(f'  ❌ {Path(f).name}: {e}')

if not all_data:
    raise ValueError('No valid data loaded from any CSV file!')

chirps_combined = pd.concat(all_data, ignore_index=True)

# ── Clean up ──
chirps_combined['date'] = pd.to_datetime(chirps_combined['date'], errors='coerce')
chirps_combined['rainfall_mm'] = pd.to_numeric(chirps_combined['rainfall_mm'], errors='coerce')
chirps_combined = chirps_combined.dropna(subset=['date', 'rainfall_mm'])

print()
print('=' * 80)
print(f'✅ COMBINED DATASET')
print(f'   Total rows:    {len(chirps_combined):>12,}')
print(f'   Districts:     {chirps_combined["district_id"].nunique():>12}')
print(f'   Date range:    {chirps_combined["date"].min()} → {chirps_combined["date"].max()}')
print(f'   Mean rainfall: {chirps_combined["rainfall_mm"].mean():.2f} mm/day')
print('=' * 80)

if errors:
    print(f'\n⚠️  {len(errors)} files had errors — check above')

# Save combined file
OUTPUT_DIR = Path('/content/historical')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
combined_path = OUTPUT_DIR / 'chirps_daily_all_districts.csv'
chirps_combined.to_csv(combined_path, index=False)
print(f'\n💾 Saved combined data: {combined_path}')
print(f'   File size: {combined_path.stat().st_size / 1024 / 1024:.1f} MB')

## Cell 7 — Compute CHIRPS Climatology Baseline

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 7: Compute day-of-year climatology from the combined data
# ══════════════════════════════════════════════════════════════════════

import numpy as np

# ── Try using the production_pipeline function ──
try:
    from production_pipeline.extractors.historical_extractor import compute_chirps_climatology
    chirps_clim = compute_chirps_climatology(chirps_combined)
    print('✅ Used production_pipeline.extractors.historical_extractor.compute_chirps_climatology')
except ImportError:
    # ── Inline fallback computation ──
    print('ℹ️  Computing climatology inline (production_pipeline not available)...')
    
    df = chirps_combined.copy()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['rainfall_mm'] = pd.to_numeric(df['rainfall_mm'], errors='coerce')
    df = df.dropna(subset=['date', 'rainfall_mm'])
    df['day_of_year'] = df['date'].dt.dayofyear
    
    # Group by district + day-of-year and compute stats
    agg = df.groupby(['district_id', 'day_of_year'])['rainfall_mm'].agg([
        ('mean_mm', 'mean'),
        ('std_mm', 'std'),
        ('p10', lambda x: np.nanpercentile(x, 10)),
        ('p25', lambda x: np.nanpercentile(x, 25)),
        ('p50', lambda x: np.nanpercentile(x, 50)),
        ('p75', lambda x: np.nanpercentile(x, 75)),
        ('p90', lambda x: np.nanpercentile(x, 90)),
    ]).reset_index()
    
    # Add district name
    name_map = chirps_combined.drop_duplicates('district_id').set_index('district_id')['district_name'].to_dict()
    agg['district_name'] = agg['district_id'].map(name_map)
    agg['source'] = f'CHIRPS_climatology_{START_YEAR}_{END_YEAR}'
    
    chirps_clim = agg
    print('✅ Computed climatology inline')

# ── Verify ──
n_districts = chirps_clim['district_id'].nunique()
n_doys = chirps_clim['day_of_year'].nunique() if 'day_of_year' in chirps_clim.columns else 0

print(f'\n📊 CLIMATOLOGY RESULTS')
print(f'   Rows:      {len(chirps_clim):,}')
print(f'   Districts: {n_districts}')
print(f'   DOYs:      {n_doys}')
print(f'   Expected:  {n_districts} × {n_doys} = {n_districts * n_doys}')

# ── Save ──
clim_path = OUTPUT_DIR / 'chirps_climatology.csv'
chirps_clim.to_csv(clim_path, index=False)
print(f'\n💾 Saved climatology: {clim_path}')
print(f'   File size: {clim_path.stat().st_size / 1024:.1f} KB')

# Also save to the bundled data directory if it exists
bundled_dir = Path('production_pipeline/data/historical')
if bundled_dir.exists():
    bundled_path = bundled_dir / 'chirps_climatology.csv'
    chirps_clim.to_csv(bundled_path, index=False)
    print(f'💾 Also saved to: {bundled_path}')

print('\n🎉 DONE! Climatology baseline is ready for the anomaly computation pipeline.')
print('   Use this file in COLAB_STEP_BY_STEP_TEST.ipynb (Step 5, mode="bundled")')

## Cell 8 — (Optional) CHIRPS Visualization

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 8: Quick sanity-check visualization
# ══════════════════════════════════════════════════════════════════════

try:
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Plot 1: Mean rainfall by DOY for a sample district
    sample_district = chirps_clim['district_id'].unique()[0]
    sample = chirps_clim[chirps_clim['district_id'] == sample_district].sort_values('day_of_year')
    sample_name = sample['district_name'].iloc[0] if 'district_name' in sample.columns else sample_district
    
    axes[0].fill_between(sample['day_of_year'], sample['p10'], sample['p90'], alpha=0.2, color='blue', label='P10–P90')
    axes[0].fill_between(sample['day_of_year'], sample['p25'], sample['p75'], alpha=0.3, color='blue', label='P25–P75')
    axes[0].plot(sample['day_of_year'], sample['mean_mm'], color='blue', linewidth=1.5, label='Mean')
    axes[0].set_xlabel('Day of Year')
    axes[0].set_ylabel('Rainfall (mm/day)')
    axes[0].set_title(f'CHIRPS Climatology — {sample_name}')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Mean annual rainfall by district
    annual_mean = chirps_clim.groupby('district_id')['mean_mm'].mean().sort_values(ascending=False)
    axes[1].barh(range(len(annual_mean)), annual_mean.values, color='steelblue', height=0.7)
    axes[1].set_xlabel('Mean Daily Rainfall (mm)')
    axes[1].set_title(f'Mean Rainfall by District ({START_YEAR}–{END_YEAR})')
    if len(annual_mean) <= 20:
        axes[1].set_yticks(range(len(annual_mean)))
        axes[1].set_yticklabels(annual_mean.index, fontsize=7)
    else:
        axes[1].set_yticks([])
    axes[1].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / 'chirps_climatology_preview.png'), dpi=100, bbox_inches='tight')
    plt.show()
    print('✅ Visualization saved')
    
except ImportError:
    print('ℹ️  matplotlib not available — skipping visualization')

---

## Cell 9 — Combine ERA5 District CSVs

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 9: Combine downloaded ERA5 district CSVs into one dataset
# ══════════════════════════════════════════════════════════════════════

import pandas as pd
import glob
from pathlib import Path

# Try multiple common locations for ERA5 files
era5_search_dirs = [
    '/content/drive/MyDrive/bangladesh_drought_historical',  # Google Drive mounted
    '/content/era5_by_district',                              # Manual upload
    '/content/bangladesh_drought_historical',                 # Direct folder
    './era5_by_district',                                     # Local
]

era5_csv_files = []
era5_source_dir = None
for d in era5_search_dirs:
    found = sorted(glob.glob(f'{d}/era5_*.csv'))
    if found:
        era5_csv_files = found
        era5_source_dir = d
        break

if not era5_csv_files:
    # Try mounting Google Drive
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        era5_csv_files = sorted(glob.glob(f'/content/drive/MyDrive/{DRIVE_FOLDER}/era5_*.csv'))
        era5_source_dir = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
    except Exception:
        pass

if not era5_csv_files:
    raise FileNotFoundError(
        '❌ No ERA5 CSV files found!\n'
        'Options:\n'
        '  1. Mount Google Drive: from google.colab import drive; drive.mount("/content/drive")\n'
        f'  2. Upload CSVs to /content/era5_by_district/\n'
        f'  3. Check that exports completed at https://code.earthengine.google.com/tasks'
    )

print(f'📂 Source: {era5_source_dir}')
print(f'📄 Found {len(era5_csv_files)} ERA5 CSV files')
print()

# ── Load and combine ──
era5_all_data = []
era5_errors = []

for f in era5_csv_files:
    try:
        df = pd.read_csv(f)
        if len(df) == 0:
            print(f'  ⚠️  Empty: {Path(f).name}')
            continue
        era5_all_data.append(df)
        district_id = df['district_id'].iloc[0] if 'district_id' in df.columns else 'unknown'
        district_name = df['district_name'].iloc[0] if 'district_name' in df.columns else 'unknown'
        print(f'  ✅ {Path(f).name}: {len(df):>8,} rows  ({district_name})')
    except Exception as e:
        era5_errors.append(f)
        print(f'  ❌ {Path(f).name}: {e}')

if not era5_all_data:
    raise ValueError('No valid data loaded from any ERA5 CSV file!')

era5_combined = pd.concat(era5_all_data, ignore_index=True)

# ── Clean up ──
era5_combined['date'] = pd.to_datetime(era5_combined['date'], errors='coerce')
era5_combined['temperature_c'] = pd.to_numeric(era5_combined['temperature_c'], errors='coerce')
era5_combined['evaporation_mm'] = pd.to_numeric(era5_combined['evaporation_mm'], errors='coerce')
era5_combined = era5_combined.dropna(subset=['date'])

print()
print('=' * 80)
print(f'✅ ERA5 COMBINED DATASET')
print(f'   Total rows:      {len(era5_combined):>12,}')
print(f'   Districts:       {era5_combined["district_id"].nunique():>12}')
print(f'   Date range:      {era5_combined["date"].min()} → {era5_combined["date"].max()}')
print(f'   Mean temp (°C):  {era5_combined["temperature_c"].mean():.2f}')
print(f'   Mean evap (mm):  {era5_combined["evaporation_mm"].mean():.4f}')
print('=' * 80)

if era5_errors:
    print(f'\n⚠️  {len(era5_errors)} files had errors — check above')

# Save combined file
OUTPUT_DIR = Path('/content/historical')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
era5_combined_path = OUTPUT_DIR / 'era5_daily_all_districts.csv'
era5_combined.to_csv(era5_combined_path, index=False)
print(f'\n💾 Saved combined ERA5 data: {era5_combined_path}')
print(f'   File size: {era5_combined_path.stat().st_size / 1024 / 1024:.1f} MB')

## Cell 10 — Compute ERA5 Climatology Baseline

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 10: Compute day-of-year ERA5 climatology from the combined data
# ══════════════════════════════════════════════════════════════════════

import numpy as np

# ── Try using the production_pipeline function ──
try:
    from production_pipeline.extractors.historical_extractor import compute_era5_climatology
    era5_clim = compute_era5_climatology(era5_combined)
    print('✅ Used production_pipeline.extractors.historical_extractor.compute_era5_climatology')
except ImportError:
    # ── Inline fallback computation ──
    print('ℹ️  Computing ERA5 climatology inline (production_pipeline not available)...')
    
    df = era5_combined.copy()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date'])
    df['day_of_year'] = df['date'].dt.dayofyear
    
    # Compute stats for temperature
    temp_cols = ['temperature_c']
    evap_cols = ['evaporation_mm']
    value_cols = [c for c in temp_cols + evap_cols if c in df.columns]
    
    results_list = []
    for col in value_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        grouped = df.groupby(['district_id', 'day_of_year'])[col]
        stats = grouped.agg(mean='mean', std='std').reset_index()
        stats['std'] = stats['std'].fillna(0.0)
        
        quantiles = (
            grouped.quantile([0.10, 0.25, 0.50, 0.75, 0.90])
            .unstack(level=-1)
            .rename(columns={0.10: 'p10', 0.25: 'p25', 0.50: 'p50', 0.75: 'p75', 0.90: 'p90'})
            .reset_index()
        )
        
        merged = stats.merge(quantiles, on=['district_id', 'day_of_year'], how='left')
        for c in ['mean', 'std', 'p10', 'p25', 'p50', 'p75', 'p90']:
            merged = merged.rename(columns={c: f'{col}_{c}'})
        results_list.append(merged)
    
    era5_clim = results_list[0]
    for r in results_list[1:]:
        era5_clim = era5_clim.merge(r, on=['district_id', 'day_of_year'], how='outer')
    
    # Add district name
    name_map = era5_combined.drop_duplicates('district_id').set_index('district_id')['district_name'].to_dict()
    era5_clim['district_name'] = era5_clim['district_id'].map(name_map)
    era5_clim['source'] = f'ERA5_climatology_{START_YEAR}_{END_YEAR}'
    
    print('✅ Computed ERA5 climatology inline')

# ── Verify ──
n_districts = era5_clim['district_id'].nunique()
n_doys = era5_clim['day_of_year'].nunique() if 'day_of_year' in era5_clim.columns else 0

print(f'\n📊 ERA5 CLIMATOLOGY RESULTS')
print(f'   Rows:      {len(era5_clim):,}')
print(f'   Districts: {n_districts}')
print(f'   DOYs:      {n_doys}')
print(f'   Expected:  {n_districts} × {n_doys} = {n_districts * n_doys}')
print(f'   Columns:   {list(era5_clim.columns)}')

# ── Save ──
era5_clim_path = OUTPUT_DIR / 'era5_climatology.csv'
era5_clim.to_csv(era5_clim_path, index=False)
print(f'\n💾 Saved ERA5 climatology: {era5_clim_path}')
print(f'   File size: {era5_clim_path.stat().st_size / 1024:.1f} KB')

# Also save to the bundled data directory if it exists
bundled_dir = Path('production_pipeline/data/historical')
if bundled_dir.exists():
    bundled_path = bundled_dir / 'era5_climatology.csv'
    era5_clim.to_csv(bundled_path, index=False)
    print(f'💾 Also saved to: {bundled_path}')

print('\n📊 Sample:')
print(era5_clim.head(10))

---

## Cell 11 — Final Summary & Verification

Verify both climatology files are ready for the anomaly computation pipeline.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 11: Final verification — both climatologies ready
# ══════════════════════════════════════════════════════════════════════

from pathlib import Path

OUTPUT_DIR = Path('/content/historical')

print('=' * 80)
print('📋 HISTORICAL DATA SUMMARY')
print('=' * 80)
print()
print('📦 What was downloaded:')
print('   ┌─────────────────────────────────────────────────────────────┐')
print('   │  CHIRPS (rainfall) ──→ SPI computation                     │')
print('   │  ERA5-Land (temp + evap) ──→ SPEI computation              │')
print('   └─────────────────────────────────────────────────────────────┘')
print()

# Check files
files_to_check = {
    'CHIRPS Daily Combined': OUTPUT_DIR / 'chirps_daily_all_districts.csv',
    'CHIRPS Climatology': OUTPUT_DIR / 'chirps_climatology.csv',
    'ERA5 Daily Combined': OUTPUT_DIR / 'era5_daily_all_districts.csv',
    'ERA5 Climatology': OUTPUT_DIR / 'era5_climatology.csv',
}

all_ok = True
for name, path in files_to_check.items():
    if path.exists():
        size_mb = path.stat().st_size / 1024 / 1024
        df_check = pd.read_csv(path, nrows=5)
        print(f'   ✅ {name}')
        print(f'      Path: {path}')
        print(f'      Size: {size_mb:.1f} MB')
        print(f'      Columns: {list(df_check.columns)}')
    else:
        print(f'   ❌ {name}: NOT FOUND at {path}')
        all_ok = False
    print()

if all_ok:
    print('🎉 ALL DONE! Both climatology baselines are ready.')
    print()
    print('📥 NEXT STEPS:')
    print('   1. Download these files from /content/historical/:')
    print('      - chirps_climatology.csv')
    print('      - era5_climatology.csv')
    print('   2. Place them in: production_pipeline/data/historical/')
    print('   3. Use in COLAB_STEP_BY_STEP_TEST.ipynb (Step 5, mode="bundled")')
    print()
    print('📊 What gets computed downstream:')
    print('   - CHIRPS climatology → SPI (drought severity from rainfall)')
    print('   - ERA5 climatology → SPEI (drought severity from water balance)')
    print('   - Both feed into the anomaly computation pipeline')
else:
    print('⚠️  Some files are missing. Check the cells above for errors.')

---

## 📋 Troubleshooting

### Q: Some tasks failed with "FAILED" status
**A:** Re-run just those districts:
```python
# For CHIRPS:
failed_ids = ['BD1004', 'BD1006']  # example
for idx, row in districts_gdf.iterrows():
    if row['district_id_canonical'] in failed_ids:
        task, desc = export_chirps_one_district(row, START_YEAR, END_YEAR, DRIVE_FOLDER)
        print(f'Re-submitted: {desc}')

# For ERA5:
for idx, row in districts_gdf.iterrows():
    if row['district_id_canonical'] in failed_ids:
        task, desc = export_era5_one_district(row, START_YEAR, END_YEAR, DRIVE_FOLDER)
        print(f'Re-submitted: {desc}')
```

### Q: How long will exports take?
**A:** 
- `recent` mode (2020–2023): ~15–30 min per dataset
- `full` mode (1981–2023): ~2–4 hours per dataset
- CHIRPS and ERA5 can run in parallel on GEE!

### Q: How do I tell CHIRPS and ERA5 files apart?
**A:** File naming convention:
- `chirps_BD1001_2020_2023.csv` → CHIRPS rainfall
- `era5_BD1001_2020_2023.csv` → ERA5 temperature + evaporation

### Q: I only got 63 CSVs instead of 64
**A:** Check the monitor cells (4b or 5b) for which district failed, then re-submit it.

### Q: The combined CSV is very large
**A:** For `full` mode (43 years × 64 districts × 365 days), expect ~1 million rows per dataset. This is normal.

### Q: What are these two datasets used for?
**A:**
- **CHIRPS** (rainfall) → Computes **SPI** (Standardized Precipitation Index)
- **ERA5-Land** (temperature + evaporation) + CHIRPS → Computes **SPEI** (Standardized Precipitation-Evapotranspiration Index)
- These are the ONLY two raw historical sources needed for the drought monitoring pipeline.

---